# Extra Trees 5 min + GOES — notebook definitivo

Arquitetura única para o TCC, usando o período **01/07/2026 a 15/08/2026**.

Compara nos mesmos **46 dias / 7.728 registros**:

- **M0** — J11 + geometria solar + céu claro + variabilidade temporal;
- **M1** — M0 + GOES-19;
- **M2** — M0 + correção residual baseada no GOES.

A validação é `GroupKFold(5)` com dias inteiros separados.  
O GOES é associado causalmente: somente a imagem mais recente no passado, nunca imagem futura.  
Há checkpoint ao fim de cada fold: se o Colab cair, rode novamente desde o início e o processamento continua do próximo fold.

Benchmark histórico de 05/09/2026:
MAE 25,4271 min/dia; RMSE 35,4709 min/dia; Bias -0,9371 min/dia; R² 0,922130; Pearson 0,960305.

In [ ]:
# 1. Instalação, imports e Drive
!pip -q install pvlib openpyxl joblib

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, re, json, math, shutil, hashlib, warnings
from collections import Counter
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pvlib, joblib

from scipy.stats import pearsonr, spearmanr
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    confusion_matrix, matthews_corrcoef, balanced_accuracy_score,
    accuracy_score, precision_score, recall_score, f1_score
)

print("Drive montado.")


In [ ]:
# 2. Configuração fixa
PIPELINE_VERSION = "DEF_1_0"
RANDOM_STATE = 2026

LAT = -22.413611
LON = -45.451944
ALT = 849
TZ = "America/Sao_Paulo"

DATA_INICIO = pd.Timestamp("2026-07-01")
DATA_FIM    = pd.Timestamp("2026-08-15")
HORA_INICIO = "05:00"
HORA_FIM    = "19:00"
REGISTROS_ESPERADOS_DIA = 168

LIMIAR_CSD3_MV = 500.0
LUX_POR_WM2_CLEAR = 120.0
TOLERANCIA_GOES_MIN = 20
GOES_ROLL_MIN = 30

N_SPLITS_OUTER = 5
N_SPLITS_INNER = 3
N_TREES_TUNE = 250
N_TREES_FOLD = 700
N_TREES_FINAL = 1000

PARAM_CANDIDATES = [
    {"min_samples_leaf": 1, "max_features": 1.0, "max_depth": None},
    {"min_samples_leaf": 2, "max_features": 1.0, "max_depth": None},
    {"min_samples_leaf": 3, "max_features": 1.0, "max_depth": None},
    {"min_samples_leaf": 1, "max_features": 0.8, "max_depth": 24},
    {"min_samples_leaf": 2, "max_features": 0.8, "max_depth": 24},
    {"min_samples_leaf": 2, "max_features": 0.7, "max_depth": 32},
]

RESIDUAL_PARAMS = {
    "n_estimators": 500,
    "min_samples_leaf": 2,
    "max_features": 0.8,
    "max_depth": 18,
}
ALPHAS_RESIDUAL = [0.50, 0.75, 1.00]

BENCHMARK = {
    "MAE_min_dia": 25.4271,
    "RMSE_min_dia": 35.4709,
    "Bias_min_dia": -0.9371,
    "R2": 0.922130,
    "Pearson_r": 0.960305,
    "Spearman_rho": 0.913167,
    "MCC": 0.89057,
}

ROOT = Path("/content/drive/MyDrive/TCC_Brilho_Solar_2026")
DIR_README = ROOT / "00_LEIA_ME"
DIR_CSD3   = ROOT / "01_DADOS_BRUTOS/02_CSD3"
DIR_GOES   = ROOT / "01_DADOS_BRUTOS/04_GOES19"
DIR_BASE   = ROOT / "02_DADOS_PROCESSADOS/01_Base_Consolidada"
DIR_VALID  = ROOT / "02_DADOS_PROCESSADOS/04_Bases_Validacao"
DIR_MODEL  = ROOT / "04_MODELOS_SALVOS/ExtraTrees/Definitivo"
DIR_RES    = ROOT / "05_RESULTADOS/ExtraTrees_5min_46dias/DEFINITIVO"
DIR_FIG    = ROOT / "06_FIGURAS/03_Figuras_Modelos/ExtraTrees_5min_GOES_Definitivo"

for d in [DIR_README,DIR_CSD3,DIR_GOES,DIR_BASE,DIR_VALID,DIR_MODEL,DIR_RES,DIR_FIG]:
    d.mkdir(parents=True, exist_ok=True)

CACHE_FEATURES = DIR_VALID / f"base_modelagem_46dias_{PIPELINE_VERSION}.csv"
CHECKPOINT = DIR_RES / f"checkpoint_outer_{PIPELINE_VERSION}.joblib"

print("Projeto:", ROOT)
print("Checkpoint:", CHECKPOINT)


In [ ]:
# 3. Localizar fontes e copiá-las para a árvore organizada
MYDRIVE = Path("/content/drive/MyDrive")

def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

BASE_ORIG = first_existing([
    MYDRIVE/"TCC/Base_Consolidada_J11_CSD3_CNR4.xlsx",
    MYDRIVE/"Atual/Base_Consolidada_J11_CSD3_CNR4.xlsx",
])
if BASE_ORIG is None:
    hits = list(MYDRIVE.rglob("Base_Consolidada_J11_CSD3_CNR4.xlsx"))
    BASE_ORIG = hits[0] if hits else None
if BASE_ORIG is None:
    raise FileNotFoundError("Base_Consolidada_J11_CSD3_CNR4.xlsx não encontrada.")

uv = list((MYDRIVE/"Atual").glob("UV_Itajub*.xlsx"))
if not uv:
    uv = list(MYDRIVE.rglob("UV_Itajub*.xlsx"))

def fim_uv(p):
    try:
        z = pd.read_excel(p, sheet_name="csd3_signal", header=1, usecols=[0])
        return pd.to_datetime(z.iloc[:,0], errors="coerce").max()
    except Exception:
        return pd.NaT

uv2 = [(fim_uv(p),p) for p in uv]
uv2 = [(d,p) for d,p in uv2 if pd.notna(d)]
uv2.sort(key=lambda x:x[0], reverse=True)
if not uv2:
    raise FileNotFoundError("UV_Itajubá*.xlsx com csd3_signal não encontrado.")
CSD3_ORIG = uv2[0][1]

goes = list((MYDRIVE/"Atual").glob("GOES19_Itajuba_acumulado_nativo*.csv"))
if not goes:
    goes = list(MYDRIVE.rglob("GOES19_Itajuba_acumulado_nativo*.csv"))
if not goes:
    raise FileNotFoundError("GOES19_Itajuba_acumulado_nativo*.csv não encontrado.")
GOES_ORIG = max(goes, key=lambda p:p.stat().st_size)

print("J11/base:", BASE_ORIG)
print("CSD3:", CSD3_ORIG, "| fim:", uv2[0][0])
print("GOES:", GOES_ORIG)

def copy_if_needed(src, dstdir):
    dst = dstdir/src.name
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        shutil.copy2(src,dst)
        print("Copiado:", dst)
    return dst

BASE_PATH = copy_if_needed(BASE_ORIG,DIR_BASE)
CSD3_PATH = copy_if_needed(CSD3_ORIG,DIR_CSD3)
GOES_PATH = copy_if_needed(GOES_ORIG,DIR_GOES)


In [ ]:
# 4. Preparação das fontes e cache da base de modelagem
MAPA_1145 = {
    1:"sup",2:"cu1",3:"cu1",4:"ci1",5:"cu2",6:"cu2",7:"ci3",8:"ci2",
    9:"ci3",10:"cu2",11:"sup",12:"st1",13:"cu2",14:"cu2",15:"mc1",
    16:"sup",17:"ci4",18:"cu2",19:"ci4",20:"cu2",21:"mc1",22:"st2",
    23:"cu2",24:"st1",25:"mc3",26:"st1",27:"mc2",28:"mc3",29:"st2",30:"mc4"
}
MAPA_1445 = {
    1:"sup",2:"cu1",3:"cu1",4:"ci1",5:"cu2",6:"cu2",7:"ci3",8:"ci2",
    9:"ci2",10:"cu1",11:"sup",12:"st1",13:"cu2",14:"ci2",15:"mc1",
    16:"sup",17:"ci4",18:"cu2",19:"ci4",20:"cu2",21:"mc1",22:"st1",
    23:"cu2",24:"st1",25:"mc3",26:"st1",27:"mc2",28:"mc3",29:"st2",30:"mc4"
}
MAPA_1745 = {
    1:"sup",2:"cu1",3:"cu1",4:"ci1",5:"cu2",6:"cu1",7:"ci3",8:"ci2",
    9:"ci3",10:"cu1",11:"sup",12:"st1",13:"cu1",14:"cu2",15:"mc2",
    16:"sup",17:"ci4",18:"cu2",19:"ci4",20:"cu2",21:"mc1",22:"st1",
    23:"cu2",24:"st1",25:"mc3",26:"st1",27:"mc2",28:"mc3",29:"st2",30:"mc4"
}

def tipo_cptec(codigo, ts_utc):
    if pd.isna(codigo) or pd.isna(ts_utc):
        return np.nan
    try:
        codigo = int(round(float(codigo)))
    except Exception:
        return np.nan
    h = pd.Timestamp(ts_utc).hour
    mapa = MAPA_1145 if h < 13 else (MAPA_1445 if h <= 17 else MAPA_1745)
    return mapa.get(codigo,np.nan)

def familia_cptec(tipo):
    if not isinstance(tipo,str):
        return np.nan
    if tipo == "sup":
        return "sup"
    for fam in ["cu","ci","st","mc"]:
        if tipo.startswith(fam):
            return fam
    return np.nan

def construir_base():
    # J11
    j11 = pd.read_excel(BASE_PATH,sheet_name="Dados_5min")
    j11["timestamp"] = pd.to_datetime(j11["timestamp_grade_5min_local"],errors="coerce")
    j11["j11_lux"] = pd.to_numeric(j11["j11_lux"],errors="coerce")
    j11 = j11[["timestamp","j11_lux"]].dropna().sort_values("timestamp").drop_duplicates("timestamp")

    # CSD3 1-min -> 5-min
    csd = pd.read_excel(CSD3_PATH,sheet_name="csd3_signal",header=1)
    csd["TIMESTAMP"] = pd.to_datetime(csd["TIMESTAMP"],errors="coerce")
    csd["signal"] = pd.to_numeric(csd["CSDSig_Avg"],errors="coerce")
    csd = csd.dropna(subset=["TIMESTAMP","signal"]).copy()
    q99 = float(csd["signal"].quantile(.99))
    csd["signal_mv"] = csd["signal"]*1000 if q99 <= 5 else csd["signal"]
    csd = csd[(csd["TIMESTAMP"]>=DATA_INICIO)&(csd["TIMESTAMP"]<DATA_FIM+pd.Timedelta(days=1))].copy()
    csd["timestamp"] = csd["TIMESTAMP"].dt.floor("5min")
    csd["brilho"] = (csd["signal_mv"]>=LIMIAR_CSD3_MV).astype(int)

    csd5 = csd.groupby("timestamp",as_index=False).agg(
        n_min_validos=("signal_mv","count"),
        n_min_brilho=("brilho","sum"),
        csd3_signal_media_mv=("signal_mv","mean")
    )
    csd5["y"] = csd5["n_min_brilho"]/csd5["n_min_validos"]
    csd5.loc[csd5["n_min_validos"]!=5,"y"] = np.nan

    df = j11.merge(csd5,on="timestamp",how="inner",validate="one_to_one")
    df["data"] = df["timestamp"].dt.normalize()
    df["hora_txt"] = df["timestamp"].dt.strftime("%H:%M")
    df = df[
        (df["data"]>=DATA_INICIO)&(df["data"]<=DATA_FIM)&
        (df["hora_txt"]>=HORA_INICIO)&(df["hora_txt"]<HORA_FIM)
    ].dropna(subset=["j11_lux","y"]).copy()

    counts = df.groupby("data").size()
    dias = counts[counts==REGISTROS_ESPERADOS_DIA].index
    df = df[df["data"].isin(dias)].copy()

    if df["data"].nunique()!=46 or len(df)!=7728:
        raise RuntimeError(f"Esperado 46 dias/7728; obtido {df['data'].nunique()}/{len(df)}")

    # Solar
    loc = pvlib.location.Location(LAT,LON,tz=TZ,altitude=ALT,name="Itajuba")
    ts = pd.DatetimeIndex(df["timestamp"]).tz_localize(TZ)
    sol = loc.get_solarposition(ts)
    clear = loc.get_clearsky(ts,model="ineichen")
    df["elevacao"] = sol["apparent_elevation"].to_numpy()
    df["azimute"] = sol["azimuth"].to_numpy()
    df["zenite"] = sol["apparent_zenith"].to_numpy()
    df["ghi_clear"] = clear["ghi"].to_numpy()
    df["lux_clear_proxy"] = (df["ghi_clear"]*LUX_POR_WM2_CLEAR).clip(lower=1)
    df["K_L"] = (df["j11_lux"]/df["lux_clear_proxy"]).clip(0,3)

    # J11 temporal causal
    df = df.sort_values(["data","timestamp"]).copy()
    gd = df.groupby("data",group_keys=False)
    df["log_lux"] = np.log1p(df["j11_lux"].clip(lower=0))
    df["lux_saturado"] = (df["j11_lux"]>=54000).astype(int)

    hd = df["timestamp"].dt.hour + df["timestamp"].dt.minute/60
    df["hora_seno"] = np.sin(2*np.pi*hd/24)
    df["hora_cosseno"] = np.cos(2*np.pi*hd/24)
    doy = df["timestamp"].dt.dayofyear
    df["doy_seno"] = np.sin(2*np.pi*doy/365.25)
    df["doy_cosseno"] = np.cos(2*np.pi*doy/365.25)
    az = np.deg2rad(df["azimute"])
    df["az_seno"] = np.sin(az)
    df["az_cosseno"] = np.cos(az)

    for lim in [6000,8000,10000,12000,14400,18000,28000]:
        df[f"lux_ge_{lim}"] = (df["j11_lux"]>=lim).astype(int)

    for nome,win in [("10",2),("15",3),("30",6),("60",12)]:
        df[f"lux_ma{nome}"] = gd["j11_lux"].transform(lambda s:s.rolling(win,min_periods=1).mean())
        df[f"lux_std{nome}"] = gd["j11_lux"].transform(lambda s:s.rolling(win,min_periods=2).std().fillna(0))
        df[f"kl_ma{nome}"] = gd["K_L"].transform(lambda s:s.rolling(win,min_periods=1).mean())
        df[f"kl_std{nome}"] = gd["K_L"].transform(lambda s:s.rolling(win,min_periods=2).std().fillna(0))
        lo = gd["j11_lux"].transform(lambda s:s.rolling(win,min_periods=1).min())
        hi = gd["j11_lux"].transform(lambda s:s.rolling(win,min_periods=1).max())
        df[f"lux_amp{nome}"] = hi-lo

    for lag,mins in [(1,5),(2,10),(3,15),(6,30),(12,60)]:
        df[f"lux_diff_{mins}"] = gd["j11_lux"].diff(lag).fillna(0)
        df[f"kl_diff_{mins}"] = gd["K_L"].diff(lag).fillna(0)

    df["lux_rel_diff_5"] = (df["lux_diff_5"]/(df["lux_ma10"].abs()+500)).clip(-5,5)

    # GOES
    gg = pd.read_csv(GOES_PATH)
    required = {"timestamp_utc","classe_central","fracao_nuvem_5x5"}
    missing = required-set(gg.columns)
    if missing:
        raise KeyError(f"Faltam colunas GOES: {sorted(missing)}")

    gg["timestamp_utc"] = pd.to_datetime(gg["timestamp_utc"],utc=True,errors="coerce")
    gg["goes_timestamp"] = gg["timestamp_utc"].dt.tz_convert(TZ).dt.tz_localize(None)
    gg["classe_central"] = pd.to_numeric(gg["classe_central"],errors="coerce")
    gg["fracao_nuvem_5x5"] = pd.to_numeric(gg["fracao_nuvem_5x5"],errors="coerce")
    gg["tipo_central"] = [tipo_cptec(c,t) for c,t in zip(gg["classe_central"],gg["timestamp_utc"])]
    gg["familia_central"] = gg["tipo_central"].apply(familia_cptec)

    valid = gg["familia_central"].notna()
    gg["goes_pixel_nuvem"] = np.where(valid,(gg["familia_central"]!="sup").astype(float),np.nan)
    gg["goes_is_cu"] = np.where(valid,(gg["familia_central"]=="cu").astype(float),np.nan)
    gg["goes_is_ci"] = np.where(valid,(gg["familia_central"]=="ci").astype(float),np.nan)
    gg["goes_is_st_mc"] = np.where(valid,gg["familia_central"].isin(["st","mc"]).astype(float),np.nan)
    gg["goes_fracao_valida"] = valid.astype(float)
    gg["goes_fcloud"] = gg["fracao_nuvem_5x5"].clip(0,1)

    gg = gg.dropna(subset=["goes_timestamp"]).sort_values("goes_timestamp").copy()
    gg = gg[
        (gg["goes_timestamp"]>=DATA_INICIO-pd.Timedelta(hours=1))&
        (gg["goes_timestamp"]<DATA_FIM+pd.Timedelta(days=1,hours=1))
    ].copy()

    gg["goes_fcloud_delta"] = gg["goes_fcloud"].diff()
    gg["goes_fcloud_absdelta"] = gg["goes_fcloud_delta"].abs()

    gr = gg.set_index("goes_timestamp")
    gg["goes_fcloud_ma30"] = gr["goes_fcloud"].rolling(f"{GOES_ROLL_MIN}min",min_periods=1).mean().to_numpy()
    gg["goes_fcloud_std30"] = gr["goes_fcloud"].rolling(f"{GOES_ROLL_MIN}min",min_periods=2).std().fillna(0).to_numpy()
    gg["goes_fcloud_min30"] = gr["goes_fcloud"].rolling(f"{GOES_ROLL_MIN}min",min_periods=1).min().to_numpy()
    gg["goes_fcloud_max30"] = gr["goes_fcloud"].rolling(f"{GOES_ROLL_MIN}min",min_periods=1).max().to_numpy()

    gcols = [
        "goes_timestamp","goes_fcloud","goes_pixel_nuvem","goes_is_cu","goes_is_ci",
        "goes_is_st_mc","goes_fracao_valida","goes_fcloud_delta","goes_fcloud_absdelta",
        "goes_fcloud_ma30","goes_fcloud_std30","goes_fcloud_min30","goes_fcloud_max30"
    ]

    df = pd.merge_asof(
        df.sort_values("timestamp"),
        gg[gcols].sort_values("goes_timestamp"),
        left_on="timestamp",right_on="goes_timestamp",
        direction="backward",
        tolerance=pd.Timedelta(minutes=TOLERANCIA_GOES_MIN)
    )
    df["goes_age_min"] = (df["timestamp"]-df["goes_timestamp"]).dt.total_seconds()/60
    df["goes_disponivel"] = df["goes_timestamp"].notna().astype(int)
    df["classe_ref"] = (df["y"]>=0.5).astype(int)

    print("Dias:",df["data"].nunique(),"| registros:",len(df))
    print(f"Cobertura GOES <= {TOLERANCIA_GOES_MIN} min: {df['goes_disponivel'].mean():.1%}")
    return df.reset_index(drop=True)

if CACHE_FEATURES.exists():
    print("Carregando cache:",CACHE_FEATURES)
    df = pd.read_csv(CACHE_FEATURES)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["data"] = pd.to_datetime(df["data"])
    if "goes_timestamp" in df.columns:
        df["goes_timestamp"] = pd.to_datetime(df["goes_timestamp"],errors="coerce")
else:
    df = construir_base()
    df.to_csv(CACHE_FEATURES,index=False)
    print("Cache salvo:",CACHE_FEATURES)

assert len(df)==7728 and df["data"].nunique()==46
print("OK — 46 dias / 7.728 registros.")


In [ ]:
# 5. Features e métricas
FEATURES_M0 = [
    "j11_lux","log_lux","lux_saturado","elevacao","zenite","az_seno","az_cosseno",
    "ghi_clear","lux_clear_proxy","K_L","hora_seno","hora_cosseno","doy_seno","doy_cosseno"
] + [f"lux_ge_{x}" for x in [6000,8000,10000,12000,14400,18000,28000]]

for w in ["10","15","30","60"]:
    FEATURES_M0 += [f"lux_ma{w}",f"lux_std{w}",f"kl_ma{w}",f"kl_std{w}",f"lux_amp{w}"]
for m in [5,10,15,30,60]:
    FEATURES_M0 += [f"lux_diff_{m}",f"kl_diff_{m}"]
FEATURES_M0 += ["lux_rel_diff_5"]

GOES_FEATURES = [
    "goes_disponivel","goes_age_min","goes_fcloud","goes_pixel_nuvem","goes_is_cu",
    "goes_is_ci","goes_is_st_mc","goes_fracao_valida","goes_fcloud_delta",
    "goes_fcloud_absdelta","goes_fcloud_ma30","goes_fcloud_std30",
    "goes_fcloud_min30","goes_fcloud_max30"
]

FEATURES_M0 = [c for c in FEATURES_M0 if c in df.columns]
FEATURES_M1 = FEATURES_M0 + [c for c in GOES_FEATURES if c in df.columns]
FEATURES_RESIDUAL = [c for c in GOES_FEATURES if c in df.columns] + [
    c for c in ["K_L","elevacao","hora_seno","hora_cosseno","lux_ma15","lux_std30","kl_ma15","kl_std30"]
    if c in df.columns
]

def corr_safe(a,b,kind="pearson"):
    a=np.asarray(a,float); b=np.asarray(b,float)
    m=np.isfinite(a)&np.isfinite(b)
    a=a[m]; b=b[m]
    if len(a)<3 or np.std(a)==0 or np.std(b)==0:
        return np.nan
    return pearsonr(a,b)[0] if kind=="pearson" else spearmanr(a,b)[0]

def daily_table(y_true,y_pred,datas):
    z=pd.DataFrame({"data":pd.to_datetime(datas),"y":np.asarray(y_true,float),"pred":np.asarray(y_pred,float)})
    return z.groupby("data",as_index=False).agg(
        CSD3_h=("y",lambda s:s.sum()*5/60),
        PRED_h=("pred",lambda s:s.sum()*5/60)
    )

def daily_metrics(y_true,y_pred,datas):
    d=daily_table(y_true,y_pred,datas)
    yt=d["CSD3_h"].to_numpy(); yp=d["PRED_h"].to_numpy()
    e=(yp-yt)*60
    return {
        "N_dias":len(d),
        "MAE_min_dia":mean_absolute_error(yt,yp)*60,
        "RMSE_min_dia":mean_squared_error(yt,yp)**0.5*60,
        "Bias_min_dia":float(e.mean()),
        "R2":r2_score(yt,yp),
        "Pearson_r":corr_safe(yt,yp,"pearson"),
        "Spearman_rho":corr_safe(yt,yp,"spearman")
    },d

def binary_metrics(y_class,pred):
    yp=(np.asarray(pred)>=.5).astype(int)
    yt=np.asarray(y_class).astype(int)
    tn,fp,fn,tp=confusion_matrix(yt,yp,labels=[0,1]).ravel()
    return {
        "MCC":matthews_corrcoef(yt,yp),
        "Balanced_accuracy":balanced_accuracy_score(yt,yp),
        "Accuracy":accuracy_score(yt,yp),
        "Precision":precision_score(yt,yp,zero_division=0),
        "Recall":recall_score(yt,yp,zero_division=0),
        "F1":f1_score(yt,yp,zero_division=0),
        "TN":int(tn),"FP":int(fp),"FN":int(fn),"TP":int(tp)
    }

def apply_cal(pred,scale=1.0,offset=0.0,zero_cut=0.0):
    p=np.clip(scale*np.asarray(pred)+offset,0,1)
    p[p<zero_cut]=0
    return p

def choose_cal(y,pred,groups):
    best=None
    for scale in [.95,1.0,1.05]:
        for offset in [-.03,0,.03]:
            for zc in [0,.03,.06]:
                pp=apply_cal(pred,scale,offset,zc)
                m,_=daily_metrics(y,pp,groups)
                key=(m["RMSE_min_dia"],m["MAE_min_dia"],abs(m["Bias_min_dia"]))
                if best is None or key<best[0]:
                    best=(key,{"scale":scale,"offset":offset,"zero_cut":zc})
    return best[1]

def fit_predict_et(Xtr,ytr,Xva,params,n_estimators,seed):
    med=Xtr.median(numeric_only=True)
    feats=[c for c in Xtr.columns if c in med.index and pd.notna(med[c])]
    model=ExtraTreesRegressor(
        n_estimators=n_estimators,random_state=seed,n_jobs=-1,
        criterion="squared_error",**params
    )
    model.fit(Xtr[feats].fillna(med[feats]),ytr)
    pred=np.clip(model.predict(Xva[feats].fillna(med[feats])),0,1)
    return model,pred,feats,med[feats]

def fit_predict_residual(Xtr,ytr,Xva,seed):
    med=Xtr.median(numeric_only=True)
    feats=[c for c in Xtr.columns if c in med.index and pd.notna(med[c])]
    model=ExtraTreesRegressor(random_state=seed,n_jobs=-1,criterion="squared_error",**RESIDUAL_PARAMS)
    model.fit(Xtr[feats].fillna(med[feats]),ytr)
    pred=np.clip(model.predict(Xva[feats].fillna(med[feats])),-.75,.75)
    return model,pred,feats,med[feats]

def dataset_signature(base):
    b=pd.util.hash_pandas_object(base[["timestamp","j11_lux","y"]],index=False).values.tobytes()
    return hashlib.sha256(b).hexdigest()[:20]

DATASET_SIGNATURE=dataset_signature(df)
print("Features M0:",len(FEATURES_M0),"| M1:",len(FEATURES_M1),"| residual:",len(FEATURES_RESIDUAL))
print("Assinatura:",DATASET_SIGNATURE)


In [ ]:
# 6. Seleção interna compacta e M2 residual
def select_inner(train,features,candidates,seed):
    X=train[features]
    y=train["y"].to_numpy(float)
    g=train["data"].astype(str).to_numpy()
    inner=GroupKFold(n_splits=min(N_SPLITS_INNER,pd.Series(g).nunique()))
    best=None

    for ip,params in enumerate(candidates):
        raw=np.full(len(train),np.nan)
        for k,(tr,va) in enumerate(inner.split(X,y,g)):
            _,p,_,_=fit_predict_et(
                X.iloc[tr],y[tr],X.iloc[va],params,N_TREES_TUNE,
                seed+ip*100+k
            )
            raw[va]=p
        cal=choose_cal(y,raw,g)
        pc=apply_cal(raw,**cal)
        met,_=daily_metrics(y,pc,g)
        key=(met["RMSE_min_dia"],met["MAE_min_dia"],abs(met["Bias_min_dia"]),-met["R2"])
        if best is None or key<best["key"]:
            best={
                "key":key,"params":params.copy(),"cal":cal.copy(),
                "inner_raw":raw.copy(),"inner_cal":pc.copy(),"metrics":met.copy()
            }
    return best

def tune_residual(train,base_oof,seed):
    resid=train["y"].to_numpy(float)-np.asarray(base_oof,float)
    avail=train["goes_disponivel"].to_numpy()>0
    if avail.sum()<100 or train.loc[avail,"data"].nunique()<5:
        return {"enabled":False,"alpha":0.0}

    br=train.loc[avail].reset_index(drop=True)
    yr=resid[avail]
    gr=br["data"].astype(str).to_numpy()
    Xr=br[FEATURES_RESIDUAL]
    inner=GroupKFold(n_splits=min(3,pd.Series(gr).nunique()))
    corr_av=np.full(len(br),np.nan)

    for k,(tr,va) in enumerate(inner.split(Xr,yr,gr)):
        _,p,_,_=fit_predict_residual(Xr.iloc[tr],yr[tr],Xr.iloc[va],seed+k)
        corr_av[va]=p

    corr_all=np.zeros(len(train))
    corr_all[np.where(avail)[0]]=corr_av

    best=None
    for alpha in ALPHAS_RESIDUAL:
        p2=np.clip(np.asarray(base_oof)+alpha*corr_all,0,1)
        m,_=daily_metrics(train["y"],p2,train["data"])
        key=(m["RMSE_min_dia"],m["MAE_min_dia"],abs(m["Bias_min_dia"]))
        if best is None or key<best[0]:
            best=(key,alpha)
    return {"enabled":True,"alpha":best[1]}


## 7. Célula principal

Esta é a única etapa demorada.  
**Não perca mais trabalho:** ao concluir cada fold, o resultado é gravado no Drive.  
Se o ambiente cair, rode novamente as células anteriores e esta célula: os folds já concluídos serão pulados.

In [ ]:
# 7. Outer GroupKFold: M0 × M1 × M2, com checkpoint
df=df.sort_values(["data","timestamp"]).reset_index(drop=True)
y_all=df["y"].to_numpy(float)
groups_all=df["data"].astype(str).to_numpy()

outer=GroupKFold(n_splits=N_SPLITS_OUTER)

if CHECKPOINT.exists():
    ck=joblib.load(CHECKPOINT)
    if ck.get("dataset_signature")!=DATASET_SIGNATURE:
        raise RuntimeError("Checkpoint incompatível com a base atual. Renomeie/apague o checkpoint antigo.")
    pred_m0_raw=ck["pred_m0_raw"]; pred_m0=ck["pred_m0"]
    pred_m1_raw=ck["pred_m1_raw"]; pred_m1=ck["pred_m1"]
    pred_m2=ck["pred_m2"]
    folds_done=set(ck["folds_done"])
    fold_records=ck["fold_records"]
    print("Retomando checkpoint. Folds concluídos:",sorted(folds_done))
else:
    pred_m0_raw=np.full(len(df),np.nan); pred_m0=np.full(len(df),np.nan)
    pred_m1_raw=np.full(len(df),np.nan); pred_m1=np.full(len(df),np.nan)
    pred_m2=np.full(len(df),np.nan)
    folds_done=set(); fold_records=[]
    print("Iniciando execução nova.")

for fold,(itr,iva) in enumerate(outer.split(df,y_all,groups_all),start=1):
    if fold in folds_done:
        print(f"Fold {fold} já concluído — pulando.")
        continue

    print("\n"+"="*88)
    print(f"FOLD {fold}/5")
    print("="*88)

    train=df.iloc[itr].reset_index(drop=True)
    val=df.iloc[iva].reset_index(drop=True)

    print("M0...")
    b0=select_inner(train,FEATURES_M0,PARAM_CANDIDATES,RANDOM_STATE+1000*fold)
    _,raw0,_,_=fit_predict_et(
        train[FEATURES_M0],train["y"].to_numpy(float),val[FEATURES_M0],
        b0["params"],N_TREES_FOLD,RANDOM_STATE+2000*fold
    )
    p0=apply_cal(raw0,**b0["cal"])

    print("M1 (+GOES)...")
    b1=select_inner(train,FEATURES_M1,PARAM_CANDIDATES,RANDOM_STATE+3000*fold)
    _,raw1,_,_=fit_predict_et(
        train[FEATURES_M1],train["y"].to_numpy(float),val[FEATURES_M1],
        b1["params"],N_TREES_FOLD,RANDOM_STATE+4000*fold
    )
    p1=apply_cal(raw1,**b1["cal"])

    print("M2 (residual GOES)...")
    ri=tune_residual(train,b0["inner_cal"],RANDOM_STATE+5000*fold)
    p2=p0.copy()

    if ri["enabled"] and ri["alpha"]>0:
        resid=train["y"].to_numpy(float)-b0["inner_cal"]
        mt=train["goes_disponivel"].to_numpy()>0
        mv=val["goes_disponivel"].to_numpy()>0
        if mt.sum()>=100 and mv.sum()>0:
            _,corr,_,_=fit_predict_residual(
                train.loc[mt,FEATURES_RESIDUAL],resid[mt],
                val.loc[mv,FEATURES_RESIDUAL],RANDOM_STATE+6000*fold
            )
            c=np.zeros(len(val)); c[mv]=corr
            p2=np.clip(p0+ri["alpha"]*c,0,1)

    pred_m0_raw[iva]=raw0; pred_m0[iva]=p0
    pred_m1_raw[iva]=raw1; pred_m1[iva]=p1
    pred_m2[iva]=p2

    met0,_=daily_metrics(val["y"],p0,val["data"])
    met1,_=daily_metrics(val["y"],p1,val["data"])
    met2,_=daily_metrics(val["y"],p2,val["data"])

    rec={
        "fold":fold,
        "dias_train":train["data"].nunique(),"dias_val":val["data"].nunique(),
        "goes_cobertura_train":float(train["goes_disponivel"].mean()),
        "goes_cobertura_val":float(val["goes_disponivel"].mean()),
        "M0_params":b0["params"],"M0_cal":b0["cal"],
        "M1_params":b1["params"],"M1_cal":b1["cal"],
        "M2_alpha":ri["alpha"],
        "M0_metrics":met0,"M1_metrics":met1,"M2_metrics":met2
    }
    fold_records.append(rec); folds_done.add(fold)

    print(f"M0: RMSE={met0['RMSE_min_dia']:.1f} | MAE={met0['MAE_min_dia']:.1f} | Bias={met0['Bias_min_dia']:.1f} | R²={met0['R2']:.3f}")
    print(f"M1: RMSE={met1['RMSE_min_dia']:.1f} | MAE={met1['MAE_min_dia']:.1f} | Bias={met1['Bias_min_dia']:.1f} | R²={met1['R2']:.3f}")
    print(f"M2: RMSE={met2['RMSE_min_dia']:.1f} | MAE={met2['MAE_min_dia']:.1f} | Bias={met2['Bias_min_dia']:.1f} | R²={met2['R2']:.3f} | alpha={ri['alpha']}")

    joblib.dump({
        "dataset_signature":DATASET_SIGNATURE,
        "pred_m0_raw":pred_m0_raw,"pred_m0":pred_m0,
        "pred_m1_raw":pred_m1_raw,"pred_m1":pred_m1,
        "pred_m2":pred_m2,
        "folds_done":sorted(folds_done),"fold_records":fold_records
    },CHECKPOINT)
    print("Checkpoint salvo:",CHECKPOINT)

assert np.isfinite(pred_m0).all() and np.isfinite(pred_m1).all() and np.isfinite(pred_m2).all()
print("\nTODOS OS FOLDS CONCLUÍDOS.")


In [ ]:
# 8. Métricas finais, ranking e exportação
preds={
    "M0 — J11 + SPA":pred_m0,
    "M1 — J11 + SPA + GOES":pred_m1,
    "M2 — M0 + residual GOES":pred_m2
}
rows=[]; daily_models={}

for nome,p in preds.items():
    md_,dd=daily_metrics(df["y"],p,df["data"])
    mb_=binary_metrics(df["classe_ref"],p)
    rows.append({"modelo":nome,**md_,**mb_})
    daily_models[nome]=dd.rename(columns={"PRED_h":nome})

ranking=pd.DataFrame(rows).sort_values(["RMSE_min_dia","MAE_min_dia"]).reset_index(drop=True)
winner=ranking.iloc[0]["modelo"]

print("RANKING OOF — 46 DIAS")
display(ranking)
print("VENCEDOR:",winner)

daily=daily_models["M0 — J11 + SPA"][["data","CSD3_h","M0 — J11 + SPA"]].copy()
for nome in ["M1 — J11 + SPA + GOES","M2 — M0 + residual GOES"]:
    daily=daily.merge(daily_models[nome][["data",nome]],on="data",how="left")

for nome in preds:
    daily[f"erro_{nome}_min"]=(daily[nome]-daily["CSD3_h"])*60
    daily[f"abs_erro_{nome}_min"]=daily[f"erro_{nome}_min"].abs()

serie5=df[[
    "timestamp","data","j11_lux","y","classe_ref","elevacao","azimute",
    "ghi_clear","K_L","goes_disponivel","goes_age_min","goes_fcloud","goes_pixel_nuvem"
]].copy()
serie5["M0_pred"]=pred_m0; serie5["M1_pred"]=pred_m1; serie5["M2_pred"]=pred_m2

fold_rows=[]
for r in fold_records:
    for mod in ["M0","M1","M2"]:
        x={"fold":r["fold"],"modelo":mod,"goes_cobertura_val":r["goes_cobertura_val"],**r[f"{mod}_metrics"]}
        if mod=="M2": x["alpha"]=r["M2_alpha"]
        fold_rows.append(x)
fold_df=pd.DataFrame(fold_rows)

xlsx=DIR_RES/"Indicadores_Modelos_5min_GOES_DEFINITIVO.xlsx"
with pd.ExcelWriter(xlsx,engine="openpyxl") as wr:
    ranking.to_excel(wr,sheet_name="Ranking_OOF",index=False)
    daily.to_excel(wr,sheet_name="Erros_Diarios",index=False)
    serie5.to_excel(wr,sheet_name="Serie_5min_OOF",index=False)
    fold_df.to_excel(wr,sheet_name="Folds",index=False)
    pd.DataFrame([BENCHMARK]).to_excel(wr,sheet_name="Benchmark_05_09",index=False)

daily.to_csv(DIR_RES/"Predicoes_Diarias_OOF.csv",index=False)
serie5.to_csv(DIR_RES/"Predicoes_5min_OOF.csv",index=False)
ranking.to_csv(DIR_RES/"Ranking_OOF.csv",index=False)

# Figura geral
plt.figure(figsize=(15,6))
plt.plot(daily["data"],daily["CSD3_h"],marker="o",lw=2,label="CSD3")
plt.plot(daily["data"],daily["M0 — J11 + SPA"],lw=1.3,label="M0 — J11 + SPA")
plt.plot(daily["data"],daily["M1 — J11 + SPA + GOES"],lw=1.5,label="M1 — + GOES")
plt.plot(daily["data"],daily["M2 — M0 + residual GOES"],lw=1.7,label="M2 — residual GOES")
plt.ylabel("Brilho solar diário (h)"); plt.xlabel("Data")
plt.title("CSD3 × Extra Trees — validação OOF nos mesmos 46 dias")
plt.grid(alpha=.22); plt.legend(); plt.xticks(rotation=45,ha="right"); plt.tight_layout()
plt.savefig(DIR_FIG/"01_CSD3_M0_M1_M2_46dias.png",dpi=220,bbox_inches="tight")
plt.show()

# Figura vencedora
rw=ranking.iloc[0]
plt.figure(figsize=(14,6))
plt.plot(daily["data"],daily["CSD3_h"],marker="o",lw=2,label="CSD3")
plt.plot(daily["data"],daily[winner],marker="o",lw=2,label=winner)
plt.ylabel("Brilho solar diário (h)"); plt.xlabel("Data")
plt.title(f"Melhor modelo OOF versus CSD3 — {winner}")
plt.grid(alpha=.22); plt.legend(); plt.xticks(rotation=45,ha="right")
txt=(f"46 dias\nMAE = {rw['MAE_min_dia']:.1f} min/dia\nRMSE = {rw['RMSE_min_dia']:.1f} min/dia\n"
     f"R² = {rw['R2']:.3f}\nr = {rw['Pearson_r']:.3f}\nBias = {rw['Bias_min_dia']:.1f} min/dia")
plt.gca().text(.015,.02,txt,transform=plt.gca().transAxes,va="bottom",
               bbox=dict(boxstyle="round",facecolor="white",alpha=.85))
plt.tight_layout()
plt.savefig(DIR_FIG/"02_Melhor_modelo_vs_CSD3.png",dpi=220,bbox_inches="tight")
plt.show()

print("Planilha:",xlsx)
print("Figuras:",DIR_FIG)


In [ ]:
# 9. Modelos finais para uso futuro
def modal_params(key):
    vals=[json.dumps(r[f"{key}_params"],sort_keys=True) for r in fold_records]
    return json.loads(Counter(vals).most_common(1)[0][0])

p0f=modal_params("M0")
p1f=modal_params("M1")
cal0f=choose_cal(y_all,pred_m0_raw,groups_all)
cal1f=choose_cal(y_all,pred_m1_raw,groups_all)

m0f,_,f0,med0=fit_predict_et(
    df[FEATURES_M0],y_all,df.iloc[:1][FEATURES_M0],p0f,N_TREES_FINAL,RANDOM_STATE+9001
)
m1f,_,f1,med1=fit_predict_et(
    df[FEATURES_M1],y_all,df.iloc[:1][FEATURES_M1],p1f,N_TREES_FINAL,RANDOM_STATE+9002
)

alpha_final=float(np.median([r["M2_alpha"] for r in fold_records]))
resid=y_all-pred_m0
mask=df["goes_disponivel"].to_numpy()>0
mrf=None; fr=[]; medr=pd.Series(dtype=float)

if mask.sum()>=100 and alpha_final>0:
    Xr=df.loc[mask,FEATURES_RESIDUAL]; yr=resid[mask]
    medr=Xr.median(numeric_only=True)
    fr=[c for c in Xr.columns if c in medr.index and pd.notna(medr[c])]
    mrf=ExtraTreesRegressor(random_state=RANDOM_STATE+9003,n_jobs=-1,criterion="squared_error",**RESIDUAL_PARAMS)
    mrf.fit(Xr[fr].fillna(medr[fr]),yr)

package={
    "M0":{"model":m0f,"features":f0,"medians":med0.to_dict(),"calibration":cal0f,"params":p0f},
    "M1":{"model":m1f,"features":f1,"medians":med1.to_dict(),"calibration":cal1f,"params":p1f},
    "M2":{"base_M0":m0f,"base_features":f0,"base_medians":med0.to_dict(),
          "base_calibration":cal0f,"residual_model":mrf,"residual_features":fr,
          "residual_medians":medr[fr].to_dict() if len(fr) else {},
          "alpha":alpha_final,"residual_params":RESIDUAL_PARAMS},
    "metadata":{
        "pipeline_version":PIPELINE_VERSION,"dataset_signature":DATASET_SIGNATURE,
        "periodo":["2026-07-01","2026-08-15"],"n_dias":46,"n_registros":7728,
        "winner_oof":winner,"ranking_oof":ranking.to_dict(orient="records"),
        "lat":LAT,"lon":LON,"alt":ALT,"tz":TZ,
        "limiar_csd3_mv":LIMIAR_CSD3_MV,"tolerancia_goes_min":TOLERANCIA_GOES_MIN
    }
}

model_path=DIR_MODEL/"ExtraTrees_5min_GOES_DEFINITIVO.joblib"
joblib.dump(package,model_path)

summary=DIR_MODEL/"RESUMO_MODELO_DEFINITIVO.txt"
with open(summary,"w",encoding="utf-8") as f:
    f.write("EXTRA TREES 5 MIN + GOES — MODELO DEFINITIVO\n")
    f.write("="*72+"\n")
    f.write(f"Pipeline: {PIPELINE_VERSION}\n")
    f.write("Período: 01/07/2026 a 15/08/2026\n")
    f.write("46 dias / 7.728 registros\n")
    f.write(f"Vencedor OOF: {winner}\n\n")
    f.write(ranking.to_string(index=False))

print("Pacote:",model_path)
print("Resumo:",summary)


## Regra de uso daqui para frente

Este é o notebook principal. Não crie novas variantes antes de concluir a comparação M0 × M1 × M2.

Se o Colab desconectar:
1. reconecte;
2. rode novamente as células 1 a 7;
3. a célula 7 lerá o checkpoint e continuará do fold seguinte.

Se alterar a metodologia de forma intencional, troque `PIPELINE_VERSION`; isso evita misturar checkpoints de versões diferentes.